# Food Delivery Time Prediction — Exploratory Data Analysis

This notebook provides a comprehensive EDA of the food delivery dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

from src.data_cleaning import clean_data
from src.feature_engineering import engineer_features

## 1. Load & Clean Data

In [ ]:
df = clean_data()
df = engineer_features(df)
print(f'Shape: {df.shape}')
df.head()

## 2. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['Time_taken(min)'].hist(bins=45, ax=axes[0], edgecolor='white', color='#3498db')
axes[0].set_title('Distribution of Delivery Time')
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Count')

df.boxplot(column='Time_taken(min)', ax=axes[1])
axes[1].set_title('Box Plot of Delivery Time')
plt.tight_layout()
plt.show()

print(df['Time_taken(min)'].describe())

## 3. Categorical Feature Analysis

In [ ]:
cat_cols = ['Weatherconditions', 'Road_traffic_density', 'Type_of_order',
            'Type_of_vehicle', 'Festival', 'City', 'order_time_period']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for i, col in enumerate(cat_cols):
    ax = axes[i // 3][i % 3]
    df.groupby(col)['Time_taken(min)'].mean().sort_values().plot(kind='barh', ax=ax, color='#2ecc71')
    ax.set_title(f'Mean Delivery Time by {col}')
    ax.set_xlabel('Minutes')
for j in range(len(cat_cols), 9):
    axes[j // 3][j % 3].axis('off')
plt.tight_layout()
plt.show()

## 4. Numerical Feature Distributions

In [ ]:
num_cols = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Vehicle_condition',
            'multiple_deliveries', 'haversine_distance_km', 'manhattan_distance_km',
            'preparation_time_min']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df[col].dropna().hist(bins=30, ax=ax, edgecolor='white', color='#e74c3c', alpha=0.7)
    ax.set_title(col)
for j in range(len(num_cols), 9):
    axes[j // 3][j % 3].axis('off')
plt.tight_layout()
plt.show()

## 5. Distance vs Delivery Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df['haversine_distance_km'], df['Time_taken(min)'], alpha=0.1, s=5)
axes[0].set_xlabel('Haversine Distance (km)')
axes[0].set_ylabel('Delivery Time (min)')
axes[0].set_title('Haversine Distance vs Delivery Time')

axes[1].scatter(df['manhattan_distance_km'], df['Time_taken(min)'], alpha=0.1, s=5)
axes[1].set_xlabel('Manhattan Distance (km)')
axes[1].set_ylabel('Delivery Time (min)')
axes[1].set_title('Manhattan Distance vs Delivery Time')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
corr_cols = num_cols + ['Time_taken(min)']
corr_matrix = df[corr_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7. Missing Values Summary

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values:')
for col, count in missing.items():
    print(f'  {col}: {count} ({count/len(df)*100:.1f}%)')

## 8. Outlier Analysis

In [ ]:
# Delivery time outlier check
q1 = df['Time_taken(min)'].quantile(0.25)
q3 = df['Time_taken(min)'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = df[(df['Time_taken(min)'] < lower) | (df['Time_taken(min)'] > upper)]
print(f'Target outliers (IQR method): {len(outliers)} rows')
print(f'  Lower bound: {lower:.1f}, Upper bound: {upper:.1f}')
print(f'  Decision: KEEP outliers — delivery times of 10-54 min are all plausible.')
print(f'  Extreme values likely reflect long-distance or high-traffic deliveries.')